# 映射类型

学习目标：能由既有属性集合生成新对象类型，控制可选性与只读性，并在重映射时保留或过滤键。

前置知识：对象类型、索引签名、keyof、索引访问类型、泛型与联合类型。

适用版本：TypeScript 7.0.2、Node.js 24.11.0；ES 模块，开启 strict；另启用 exactOptionalPropertyTypes。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/13-mapped-types/。

1. [main.ts](scripts/13-mapped-types/main.ts)：配套实现与示例。
2. [tsconfig.json](scripts/13-mapped-types/tsconfig.json)：本章独立项目配置。
3. [type-errors.ts](scripts/13-mapped-types/type-errors.ts)、[tsconfig.errors.json](scripts/13-mapped-types/tsconfig.errors.json)：单独检查的类型反例。

Step 1：检查本章正常示例的类型。

```bash
npm run check:13
```

Step 2：生成本章 JavaScript。

```bash
npm run build:13
```

Step 3：运行本章正常示例。

```bash
npm run run:13
# 正常退出；各段预期输出见代码注释。
```

正常片段均按正文顺序节选自 main.ts，前文定义在后续片段中继续使用。直接运行完整项目；反例使用独立配置，不进入正常运行入口。

## 1 遍历键并变换属性类型

映射类型（mapped type）根据键的集合生成对象类型，不是运行时循环。下面 T 是输入对象类型，P 是逐个取出的属性键；把每个属性的值类型统一改成 boolean，可以表示字段是否通过检查。

这里仅在类型层构造 Flag 类型，真正的布尔值仍由程序或输入提供。

```typescript
export type Flags<T> = { [P in keyof T]: boolean };
type Form = { title: string; hours: number };
const flags: Flags<Form> = { title: true, hours: false };
console.log(flags.title, flags.hours);
// 预期输出：true false
```

## 2 保留属性信息与索引签名的区别

按 keyof T 映射并返回 T[P] 时，通常称为同态映射（homomorphic mapped type）：属性来自原类型，未改变的 readonly 和可选标记可以保留。

有限键映射要求列出的必需键齐全；字符串索引签名表达任意字符串键的值约束，不会枚举出必须存在的 name 和 count。二者语法相似，承诺不同。

```typescript
export type Copy<T> = { [P in keyof T]: T[P] };
type Source = { readonly id: number; note?: string };
const copied: Copy<Source> = { id: 7 };
type ExactFlags = { [P in "name" | "count"]: boolean };
type OpenFlags = { [key: string]: boolean };
const exact: ExactFlags = { name: true, count: false };
const open: OpenFlags = {};
console.log(copied.id, copied.note, exact.name, Object.keys(open).length);
// 预期输出：7 undefined true 0
```

## 3 添加和移除 readonly

映射修饰符可以改变属性的写入许可。readonly 或 +readonly 添加只读标记，-readonly 移除它；没有符号时默认是添加。

这仍是静态视图，既不会冻结对象，也不会复制对象。浅层只读仅限制直接属性重新赋值，嵌套对象要按其自身类型判断。

```typescript
export type Locked<T> = { readonly [P in keyof T]: T[P] };
export type Mutable<T> = { -readonly [P in keyof T]: T[P] };
const locked: Locked<{ nested: { count: number } }> = { nested: { count: 1 } };
locked.nested.count += 1;
const editable: Mutable<Source> = { id: 7 };
editable.id = 8;
console.log(locked.nested.count, editable.id);
// 预期输出：2 8
```

## 4 添加和移除可选标记

属性后的 ? 或 +? 使键可缺省，-? 要求键存在。本章显式启用 exactOptionalPropertyTypes，以区分“键没出现”和“键存在但值为 undefined”。

必需化不会填入默认值。输入类型若明确允许 undefined，去掉可选标记后仍可以保留这种值；需要运行时默认值时必须自行赋值。

```typescript
export type Optional<T> = { [P in keyof T]?: T[P] };
export type Complete<T> = { [P in keyof T]-?: T[P] };
const patch: Optional<Form> = { title: "新标题" };
const complete: Complete<{ note?: string }> = { note: "已填" };
const explicit: Complete<{ note?: string | undefined }> = { note: undefined };
console.log(patch.title, complete.note, explicit.note);
// 预期输出：新标题 已填 undefined
```

## 5 用 as 重映射和过滤键

映射中的 as 决定新键，不是值的类型断言。先用已有的键名字典重新命名，可以在尚未组合模板字符串时看清映射关系：P 取旧键，Names[P] 给出新键，T[P] 保留原属性值的类型。

把某个键映射成 never 会过滤该键。这里 Rename.id 明确写成 never，避免提前把条件类型的推导混入主线；下一章再介绍按条件得到 never。模板式批量改名在模板字面量类型章节展开。

```typescript
type Names = { title: "caption"; hours: "duration" };
type Renamed<T extends { title: unknown; hours: unknown }> = {
  [P in keyof Names as Names[P]]: T[P]
};
const renamed: Renamed<Form> = { caption: "映射", duration: 2 };
type Rename = { id: never; title: "title" };
type Stored = { id: number; title: string };
export type PublicStored = { [P in keyof Stored as Rename[P]]: Stored[P] };
const publicValue: PublicStored = { title: "可见" };
console.log(renamed.caption, renamed.duration, publicValue.title);
// 预期输出：映射 2 可见
```

## 6 检查类型边界

下面的 [type-errors.ts](scripts/13-mapped-types/type-errors.ts) 只用于检查，不执行。逐项阅读注释，修正时保留原本需求，不通过断言或关闭检查掩盖错误。

```typescript
import type { Copy, Complete, Optional, PublicStored } from "./main.js";
const preserved: Copy<{ readonly id: number }> = { id: 1 };
preserved.id = 2; // 映射保留了 readonly。
const absent: Complete<{ title?: string }> = {}; // 去掉 ? 后必须提供 title。
const explicit: Optional<{ title: string }> = { title: undefined }; // 精确可选属性检查不允许这个值。
const leaked: PublicStored = { title: "公开", id: 1 }; // id 已被过滤，直接字面量触发额外属性检查。
// 预期诊断包含：TS2540, TS2741, TS2375, TS2353。
```

Step 1：单独检查反例并对照错误位置与原因。

```bash
npm run errors:13
# 本章固定编译器预期退出码为 1；正常项目命令的退出码为 0。
```

## 本章小结

映射类型按键构造新类型；默认保留的信息、修饰符的增减、新键的选择需要分别考虑。映射不创建值、填默认值或冻结嵌套对象。

## 练习

1. 为 Form 添加 active: boolean，再创建 Flags\<Form\>；确认缺少 active 时检查失败，补齐后通过。

2. 组合 -readonly 和 -? 定义可编辑且必需的类型；核对可以写入 id，但不能省略 note。

3. 将 Rename.title 改为 caption，调整对象值；确认旧属性 title 被拒绝，新属性 caption 被接受。

## 参考与引用来源

- TypeScript 官方文档：[Mapped Types：Mapping Modifiers、Key Remapping via as](https://www.typescriptlang.org/docs/handbook/2/mapped-types.html)；[2.8：Improved control over mapped type modifiers](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-2-8.html#improved-control-over-mapped-type-modifiers)；[exactOptionalPropertyTypes](https://www.typescriptlang.org/tsconfig/exactOptionalPropertyTypes.html)；[readonly Properties 与 Index Signatures](https://www.typescriptlang.org/docs/handbook/2/objects.html)。